# Experiment on paraphrase generalisation

In [1]:
import sys
sys.path.insert(1, "C:/Users/hp/Downloads/RL-X/one_policy_to_run_them_all/sentence_transformer")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import csv
from semanticreasoning import ProcessData
from neuralnetwork import NeuralNetwork

c:\ProgramData\anaconda3\envs\robotics\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#loading trainset

with open("../commands.csv", "r") as f:
    reader = csv.reader(f)

    next(reader)
    texts = []
    commands = []
    for row in reader:

        if not any(row):
            continue
        texts.append(row[0])
        row_command = []
        for others in row[1:4]:
            row_command.append(float(others))
        commands.append(row_command)

In [3]:
dataProcessor = ProcessData()
embeddings = dataProcessor.getEmbeddings(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1602.46it/s]


In [4]:
class MyDataSet(Dataset):
    def __init__(self,texts, embeddings, commands):
        self.texts = texts
        self.commands = commands
        self.embeddings = embeddings

    def __len__(self):
        if len(self.texts) == len(self.commands):
            return len(self.texts)
        else:
            raise Exception("The input and output lenght should be equal")
        
    def __getitem__(self, index):
        return self.texts[index], self.embeddings, self.commands[index]

In [5]:
trainset = MyDataSet(texts, embeddings, commands)
trainset[0]

('move forward',
 tensor([[-0.0389, -0.0313,  0.0001,  ..., -0.1031,  0.0016,  0.0062],
         [-0.0198, -0.0071, -0.0511,  ..., -0.0710,  0.0544, -0.0073],
         [-0.0228, -0.0136, -0.0153,  ..., -0.0373,  0.0078,  0.0161],
         ...,
         [-0.0199,  0.0207,  0.0029,  ..., -0.0405, -0.0558,  0.0226],
         [-0.0090,  0.0362, -0.0073,  ...,  0.0271, -0.0552,  0.0437],
         [ 0.0360, -0.0558,  0.0674,  ...,  0.0082, -0.0825,  0.0280]]),
 [1.0, 0.0, 0.0])

In [6]:
#command classes 
command_class = {}

command_class['forward'] = [1,0,0]
command_class['backward'] = [-1, 0, 0]
command_class['strafe_left'] = [0,-1,0]
command_class['strafe_right'] = [0,1,0]
command_class['turn_left']=[0,0,-1]
command_class['turn_right']=[0,0,1]
command_class['slow_forward']=[0.4,0,0]
command_class['fast_forward']=[1,0,0]
command_class['slight_left_turn']=[0,0,-0.4]
command_class['slight_right_turn']=[0,0,0.4]
command_class['forward_turn_left']=[0.7,0,-0.5]
command_class['forward_turn_right']=[0.7,0,0.5]
command_class['diagonal_forward_left']=[0.7,-0.5,0]
command_class['diagonal_forward_right']=[0.7,0.5,0]
command_class['stop']=[0,0,0]

In [7]:
#loading test set 

with open("../test_commands.csv", "r") as t:
    text_reader = csv.reader(t)
    next(text_reader)

    testcommands = text_reader
    unseen_text = []
    c_class = [] #command class
    command = []
    difficulty = []
    for row in testcommands:
        if row[1] in list(command_class.keys()):
            unseen_text.append(row[0])
            c_class.append(row[1])
            command.append(command_class[row[1]])
            difficulty.append(row[2])


In [8]:
test_embeddings = dataProcessor.getEmbeddings(unseen_text)

In [9]:
trainset = MyDataSet(unseen_text, test_embeddings, command)

In [10]:
input_size = embeddings.shape[1]
output_size = len(commands[0])

In [11]:
model = nn.Sequential(
    nn.Linear(input_size, 74),
    nn.ReLU(),
    nn.Dropout(0.224),
    nn.Linear(74, output_size)
)

In [12]:
#training

In [13]:
mlp = NeuralNetwork(inputSize=input_size, outputSize=output_size, useModel=True, model = model )

In [14]:
embeddings = torch.tensor(embeddings)

C:\Users\hp\AppData\Local\Temp\ipykernel_6072\1479695680.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embeddings = torch.tensor(embeddings)


In [15]:
commands = torch.tensor(commands)

In [16]:
mlp.trainModel(embeddings, commands)

Epoch [100/1000], Loss: 0.0216
Epoch [200/1000], Loss: 0.0102
Epoch [300/1000], Loss: 0.0086
Epoch [400/1000], Loss: 0.0092
Epoch [500/1000], Loss: 0.0077
Epoch [600/1000], Loss: 0.0076
Epoch [700/1000], Loss: 0.0075
Epoch [800/1000], Loss: 0.0072
Epoch [900/1000], Loss: 0.0066
Epoch [1000/1000], Loss: 0.0072


In [17]:
test_embeddings = torch.tensor(test_embeddings)
command = torch.tensor(command)

C:\Users\hp\AppData\Local\Temp\ipykernel_6072\2243899332.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_embeddings = torch.tensor(test_embeddings)


In [18]:
mse = nn.MSELoss()
mae = nn.L1Loss()

In [19]:
output = mlp._predict(test_embeddings)

In [20]:
mse_values, mae_values = mlp.testModel(output, command)
mse_values.mean(dim=1).shape

torch.Size([180])

In [21]:
mse_values = mse_values.mean(dim=1)
mae_values = mae_values.mean(dim=1)


In [22]:
resultdata = list(zip(mse_values,mae_values,c_class,difficulty))

In [23]:
_keys = []
results = {}

for classes in difficulty:
    if classes not in _keys:
        _keys.append(classes)

for k in _keys:
    mse_list = []
    mae_list = []
    for row in resultdata:
        if row[3] == k:
            mse_list.append(row[0])
            mae_list.append(row[1])
    results[k] = [torch.mean(torch.tensor(mse_list)), torch.mean(torch.tensor(mae_list))]
results

{'Simple paraphrase': [tensor(0.0767), tensor(0.1863)],
 'Natural paraphrase': [tensor(0.1280), tensor(0.2211)],
 'Conversational': [tensor(0.0952), tensor(0.2017)],
 'Indirect': [tensor(0.1863), tensor(0.2820)]}

In [30]:
#checking the output
sentence = "Turn to the right"
vector = dataProcessor.getEmbeddings(sentence)

In [31]:
vector = torch.tensor(vector)
vector = vector.unsqueeze(0)

C:\Users\hp\AppData\Local\Temp\ipykernel_6072\1951575885.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  vector = torch.tensor(vector)


In [32]:
mlp.model.eval()

with torch.no_grad():
    output = model(vector)

print(output)

tensor([[-0.0177,  0.0301,  0.9741]])
